# Kaggle Runtime — EDA và tiền xử lý Fraud-XAI

## 1. Thư viện, Kaggle Input và CUDA

In [ ]:
from pathlib import Path
import gc
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
if hasattr(pd.options.mode, "copy_on_write"):
    pd.options.mode.copy_on_write = True
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

KAGGLE_WORKING_ROOT = Path("/kaggle/working")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_WORKING_ROOT.is_dir() and KAGGLE_INPUT_ROOT.is_dir()

if sys.version_info < (3, 10):
    raise RuntimeError(
        f"Notebook yêu cầu Python >= 3.10; kernel hiện tại là {sys.version.split()[0]}."
    )

if IS_KAGGLE:
    # Kaggle Input đã được gắn trực tiếp; không clone Git hoặc sao chép raw data.
    PROJECT_ROOT = KAGGLE_WORKING_ROOT / "FAIR_2026_Experiment"
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    # Local: kernel có thể khởi chạy tại workspace hoặc thư mục Notebook.
    current_dir = Path.cwd().resolve()
    local_candidates = (current_dir, current_dir.parent)
    PROJECT_ROOT = next(
        (path for path in local_candidates if (path / "data").is_dir() and (path / "Notebook").is_dir()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Không xác định được workspace từ thư mục chạy hiện tại. "
            "Hãy mở kernel tại FAIR_2026_Experiment hoặc FAIR_2026_Experiment/Notebook."
        )

RELATIVE_DATA_FILES = {
    "creditcard.csv": Path("data/Raw_data/MLG_ULB/creditcard.csv"),
    "train_transaction.csv": Path("data/Raw_data/ieee-fraud-detection/train_transaction.csv"),
    "train_identity.csv": Path("data/Raw_data/ieee-fraud-detection/train_identity.csv"),
    "fraudTrain.csv": Path("data/Raw_data/Sparkov/fraudTrain.csv"),
    "fraudTest.csv": Path("data/Raw_data/Sparkov/fraudTest.csv"),
}
KAGGLE_DATA_HINTS = {
    "creditcard.csv": KAGGLE_INPUT_ROOT / "mlg-ulb-creditcardfraud" / "creditcard.csv",
    "train_transaction.csv": KAGGLE_INPUT_ROOT / "ieee-fraud-detection" / "train_transaction.csv",
    "train_identity.csv": KAGGLE_INPUT_ROOT / "ieee-fraud-detection" / "train_identity.csv",
    "fraudTrain.csv": KAGGLE_INPUT_ROOT / "fraud-detection" / "fraudTrain.csv",
    "fraudTest.csv": KAGGLE_INPUT_ROOT / "fraud-detection" / "fraudTest.csv",
}
DATA_ROOT = PROJECT_ROOT / "data" / "Raw_data"

def find_data_file(filename):
    """Kaggle ưu tiên Input đã gắn; local dùng đường dẫn tương đối trong workspace."""
    if filename not in RELATIVE_DATA_FILES:
        raise KeyError(f"Chưa khai báo đường dẫn tương đối cho {filename!r}")

    if IS_KAGGLE:
        hinted_path = KAGGLE_DATA_HINTS[filename]
        if hinted_path.is_file():
            return hinted_path
        matches = list(KAGGLE_INPUT_ROOT.rglob(filename))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            paths = "\n".join(str(path) for path in matches)
            raise RuntimeError(f"Có nhiều file Kaggle Input cùng tên {filename!r}:\n{paths}")

    workspace_path = PROJECT_ROOT / RELATIVE_DATA_FILES[filename]
    if workspace_path.is_file():
        return workspace_path

    raise FileNotFoundError(
        f"Không tìm thấy {filename!r} tại {workspace_path}"
        + (" hoặc trong /kaggle/input" if IS_KAGGLE else "")
    )

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Python: {sys.version.split()[0]}")
print(f"Output root: {PROJECT_ROOT}")

def summarize_dataset(df, dataset_name, target=None):
    """Tóm tắt schema/thiếu dữ liệu/nhãn mà không thay đổi DataFrame."""
    print(f"\n{'=' * 24} {dataset_name} {'=' * 24}")
    print(f"Shape: {df.shape[0]:,} dòng × {df.shape[1]:,} cột")
    print(f"Số dòng trùng hoàn toàn: {df.duplicated().sum():,}")

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    datetime_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
    identifier_cols = [
        col for col in df.columns
        if col.lower() in {"transactionid", "cc_num", "trans_num", "unnamed: 0"}
        or col.lower().endswith("_id")
    ]

    print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
    print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
    print(f"Datetime đã parse ({len(datetime_cols)}): {datetime_cols}")
    print(f"Identifier-like cần xem xét ({len(identifier_cols)}): {identifier_cols}")

    missing = df.isna().sum()
    missing = missing[missing.gt(0)].sort_values(ascending=False)
    missing_report = pd.DataFrame({
        "missing_count": missing,
        "missing_rate_pct": (missing / len(df) * 100).round(4),
    })
    print(f"Cột có giá trị thiếu: {len(missing_report):,}/{df.shape[1]:,}")
    if not missing_report.empty:
        display(missing_report.head(20))

    if target is not None:
        if target not in df.columns:
            raise KeyError(f"Không tìm thấy target {target!r} trong {dataset_name}")
        target_counts = df[target].value_counts(dropna=False).sort_index()
        target_rates = (df[target].value_counts(dropna=False, normalize=True).sort_index() * 100).round(6)
        target_report = pd.DataFrame({"count": target_counts, "rate_pct": target_rates})
        print(f"Phân bố target: {target}")
        display(target_report)

        ax = target_counts.plot(kind="bar", logy=True, color=["#4C78A8", "#E45756"])
        ax.set_title(f"{dataset_name} — phân bố {target} (log scale)")
        ax.set_xlabel(target)
        ax.set_ylabel("Số mẫu")
        plt.tight_layout()
        plt.show()

    if not missing_report.empty:
        top_missing = missing_report.head(20).sort_values("missing_rate_pct")
        ax = top_missing["missing_rate_pct"].plot(kind="barh", color="#F58518")
        ax.set_title(f"{dataset_name} — 20 cột thiếu nhiều nhất")
        ax.set_xlabel("Tỷ lệ thiếu (%)")
        plt.tight_layout()
        plt.show()

    display(df.head())
    return missing_report

def plot_amount_by_target(df, amount_col, target, dataset_name):
    """Quan sát phân phối số tiền theo nhãn; log1p chỉ dùng để vẽ, không tạo feature."""
    if amount_col not in df.columns or target not in df.columns:
        return
    fig, ax = plt.subplots(figsize=(9, 4))
    for label, color in [(0, "#4C78A8"), (1, "#E45756")]:
        values = pd.to_numeric(
            df.loc[df[target].eq(label), amount_col], errors="coerce"
        ).dropna()
        values = values[values.ge(0)]
        ax.hist(np.log1p(values), bins=60, alpha=0.55, density=True, label=str(label), color=color)
    ax.set_title(f"{dataset_name} — phân phối log1p({amount_col}) theo {target}")
    ax.set_xlabel(f"log1p({amount_col}) (chỉ phục vụ trực quan hóa)")
    ax.set_ylabel("Mật độ")
    ax.legend(title=target)
    plt.tight_layout()
    plt.show()

In [ ]:
# GPU/Kaggle Cell — nhận diện T4 x2 và lấy đường dẫn dữ liệu dùng chung.
import os
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

try:
    import torch
except ImportError:
    torch = None

CUDA_AVAILABLE = torch is not None and torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if CUDA_AVAILABLE else 0
GPU_IDS = list(range(min(GPU_COUNT, 2)))
GPU_NAMES = [torch.cuda.get_device_name(index) for index in GPU_IDS] if CUDA_AVAILABLE else []
IS_T4_X2 = len(GPU_NAMES) == 2 and all("T4" in name.upper() for name in GPU_NAMES)
MODEL_DEVICE = torch.device("cuda:0") if CUDA_AVAILABLE else (torch.device("cpu") if torch else "cpu")
USE_MULTI_GPU = GPU_COUNT >= 2
USE_MIXED_PRECISION = CUDA_AVAILABLE
MIXED_PRECISION_DTYPE = torch.float16 if CUDA_AVAILABLE else None
PIN_MEMORY = CUDA_AVAILABLE
DATA_LOADER_WORKERS = min(4, os.cpu_count() or 1)

if CUDA_AVAILABLE:
    # Giữ tính tái lập; mixed precision sẽ được dùng ở cell train PyTorch/TabNet sau này.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.cuda.empty_cache()

RESOLVED_DATA_FILES = {
    filename: find_data_file(filename)
    for filename in [
        "creditcard.csv", "train_transaction.csv", "train_identity.csv",
        "fraudTrain.csv", "fraudTest.csv",
    ]
}

print(f"Runtime: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"CUDA available: {CUDA_AVAILABLE} | GPU count: {GPU_COUNT} | GPU names: {GPU_NAMES}")
print(f"T4 x2 detected: {IS_T4_X2} | primary device: {MODEL_DEVICE}")
print(f"Multi-GPU ready: {USE_MULTI_GPU} | mixed precision ready: {USE_MIXED_PRECISION}")
for filename, path in RESOLVED_DATA_FILES.items():
    print(f"{filename}: {path}")

## 2. Phân tích dữ liệu (EDA)

Mục tiêu: Phân tích tính chất, quan sát dữ liệu các mẫu dữ liệu lớn trước khi đưa vào pipeline tiền xử lý

### Bộ 1: MLG_ULB

In [ ]:
mlg_path = find_data_file("creditcard.csv")
print(f"Đọc MLG-ULB từ: {mlg_path}")
mlg_df = pd.read_csv(mlg_path)

mlg_missing = summarize_dataset(mlg_df, "MLG-ULB", target="Class")
plot_amount_by_target(mlg_df, "Amount", "Class", "MLG-ULB")

expected_mlg_columns = ["Time", *[f"V{i}" for i in range(1, 29)], "Amount", "Class"]
print("Schema đúng cấu trúc MLG-ULB kỳ vọng:", list(mlg_df.columns) == expected_mlg_columns)

# Giải phóng RAM; preprocessing sẽ đọc lại đúng file khi tới phần xử lý.
del mlg_df
gc.collect()

### Bộ 2: IEEE-FRAUD-DETECTION

In [ ]:
ieee_transaction_path = find_data_file("train_transaction.csv")
ieee_identity_path = find_data_file("train_identity.csv")
print(f"Đọc IEEE transaction từ: {ieee_transaction_path}")
print(f"Đọc IEEE identity từ: {ieee_identity_path}")

ieee_transaction_df = pd.read_csv(ieee_transaction_path)
ieee_identity_df = pd.read_csv(ieee_identity_path)

ieee_transaction_missing = summarize_dataset(
    ieee_transaction_df, "IEEE-CIS train_transaction", target="isFraud"
)
ieee_identity_missing = summarize_dataset(
    ieee_identity_df, "IEEE-CIS train_identity"
)
plot_amount_by_target(
    ieee_transaction_df, "TransactionAmt", "isFraud", "IEEE-CIS train_transaction"
)

# Chỉ kiểm tra quan hệ khóa; chưa merge để giữ nguyên protocol đang cần xác minh.
transaction_ids = ieee_transaction_df["TransactionID"]
identity_ids = ieee_identity_df["TransactionID"]
print(f"TransactionID duy nhất ở transaction: {transaction_ids.nunique():,}/{len(transaction_ids):,}")
print(f"TransactionID duy nhất ở identity: {identity_ids.nunique():,}/{len(identity_ids):,}")
print(f"Identity ID khớp transaction: {identity_ids.isin(transaction_ids).sum():,}/{len(identity_ids):,}")
print("IEEE hiện được quan sát dưới dạng hai bảng riêng; preprocessing sẽ LEFT JOIN theo đặc tả.")

# IEEE lớn: giải phóng hai bảng EDA trước khi sang dataset kế tiếp.
del ieee_transaction_df, ieee_identity_df, transaction_ids, identity_ids
gc.collect()

### Bộ 3: Sparkov

In [ ]:
sparkov_train_path = find_data_file("fraudTrain.csv")
sparkov_test_path = find_data_file("fraudTest.csv")
print(f"Đọc Sparkov train từ: {sparkov_train_path}")
print(f"Đọc Sparkov test từ: {sparkov_test_path}")

sparkov_train_df = pd.read_csv(sparkov_train_path)
sparkov_test_df = pd.read_csv(sparkov_test_path)

sparkov_train_missing = summarize_dataset(
    sparkov_train_df, "Sparkov fraudTrain", target="is_fraud"
)
sparkov_test_missing = summarize_dataset(
    sparkov_test_df, "Sparkov fraudTest", target="is_fraud"
)
plot_amount_by_target(sparkov_train_df, "amt", "is_fraud", "Sparkov fraudTrain")

schema_difference = sorted(set(sparkov_train_df.columns).symmetric_difference(sparkov_test_df.columns))
print(f"Khác biệt schema giữa fraudTrain và fraudTest: {schema_difference}")
print("Các cột thời gian hiện vẫn ở dạng raw; notebook EDA chưa tạo derived feature.")

# Giải phóng RAM trước phần preprocessing.
del sparkov_train_df, sparkov_test_df
gc.collect()

# Tiền xử lý và xuất dữ liệu CSV

In [ ]:
# Cell chung — dependency, thư mục output và hàm lưu từng fold.
import inspect

# Chỉ đổi giá trị này để cả 3 dataset chạy cùng một fold.
SELECTED_FOLD = 1
if SELECTED_FOLD not in range(1, 6):
    raise ValueError("SELECTED_FOLD phải nằm trong [1, 2, 3, 4, 5].")

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from imblearn.over_sampling import SMOTE
except ImportError as exc:
    raise ImportError("Kernel thiếu imbalanced-learn; notebook không tự cài dependency.") from exc

PROCESSED_DATA_ROOT = PROJECT_ROOT / "data" / "Processed_data"
DATASET_OUTPUT_DIRS = {
    "MLG_ULB": PROCESSED_DATA_ROOT / "MLG_ULB" / "reproduction",
    "IEEE_CIS": PROCESSED_DATA_ROOT / "IEEE_CIS" / "reproduction",
    "Sparkov": PROCESSED_DATA_ROOT / "Sparkov" / "reproduction",
}
for output_dir in DATASET_OUTPUT_DIRS.values():
    output_dir.mkdir(parents=True, exist_ok=True)
OVERWRITE_PROCESSED_FOLDS = False

def make_dense_one_hot_encoder():
    kwargs = {"handle_unknown": "ignore"}
    if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
        kwargs["sparse_output"] = False
    else:
        kwargs["sparse"] = False
    return OneHotEncoder(**kwargs)

def save_dataset_definition(output_dir, config, fold_indices):
    output_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(
        [{"parameter": key, "value": value} for key, value in config.items()]
    ).to_csv(output_dir / "config.csv", index=False)
    row_count = max(int(np.max(valid_idx)) for _, valid_idx in fold_indices) + 1
    validation_fold = np.zeros(row_count, dtype=np.int8)
    for fold_number, (train_idx, valid_idx) in enumerate(fold_indices, 1):
        validation_fold[np.asarray(valid_idx)] = fold_number
    pd.DataFrame({
        "source_index": np.arange(row_count),
        "validation_fold": validation_fold,
    }).to_csv(output_dir / "fold_assignments.csv", index=False)

CSV_GZIP_OPTIONS = {"method": "gzip", "compresslevel": 3}
CSV_WRITE_CHUNKSIZE = 25_000

def get_fold_csv_paths(output_dir, fold_number):
    return {
        "train": output_dir / f"train_fold_{fold_number:02d}.csv.gz",
        "validation": output_dir / f"validation_fold_{fold_number:02d}.csv.gz",
        "reference": output_dir / f"validation_reference_fold_{fold_number:02d}.csv",
        "metadata": output_dir / f"metadata_fold_{fold_number:02d}.csv",
        "success": output_dir / f"success_fold_{fold_number:02d}.csv",
    }

def fold_csv_is_complete(paths):
    return all(paths[key].is_file() for key in ["train", "validation", "reference", "metadata", "success"])

FOLD_OUTPUT_PATTERNS = [
    "train_fold_*.csv", "train_fold_*.csv.gz", "train_fold_*.csv.gz.tmp",
    "validation_fold_*.csv", "validation_fold_*.csv.gz", "validation_fold_*.csv.gz.tmp",
    "validation_reference_fold_*.csv", "metadata_fold_*.csv", "success_fold_*.csv",
]

def prepare_single_fold_storage(output_dir, fold_number, paths):
    """Chỉ giữ một fold trong output_dir; chỉ xóa output sinh ra, không xóa raw/config."""
    output_dir.mkdir(parents=True, exist_ok=True)
    processed_root = PROCESSED_DATA_ROOT.resolve()
    resolved_output = output_dir.resolve()
    if resolved_output != processed_root and processed_root not in resolved_output.parents:
        raise RuntimeError(f"Từ chối dọn file ngoài Processed_data: {resolved_output}")

    selected_token = f"fold_{fold_number:02d}"
    for pattern in FOLD_OUTPUT_PATTERNS:
        for generated_path in output_dir.glob(pattern):
            if generated_path.is_file() and selected_token not in generated_path.name:
                generated_path.unlink()
                print(f"Đã xóa fold cũ: {generated_path.name}")

    if fold_csv_is_complete(paths) and not OVERWRITE_PROCESSED_FOLDS:
        return False

    # Fold đang chọn chưa hoàn tất hoặc được yêu cầu ghi lại: xóa phần dở trước khi chạy.
    for pattern in FOLD_OUTPUT_PATTERNS:
        for generated_path in output_dir.glob(pattern):
            if generated_path.is_file() and selected_token in generated_path.name:
                generated_path.unlink()
    return True

def write_csv_gzip_atomic(df, output_path):
    temp_path = output_path.with_name(output_path.name + ".tmp")
    if temp_path.exists():
        temp_path.unlink()
    df.to_csv(
        temp_path, index=False, compression=CSV_GZIP_OPTIONS,
        float_format="%.7g", chunksize=CSV_WRITE_CHUNKSIZE,
    )
    temp_path.replace(output_path)

print(f"Selected fold for all datasets: {SELECTED_FOLD}")
print(f"Processed data root: {PROCESSED_DATA_ROOT}")

### Bộ 1: MLG_ULB

In [ ]:
# MLG Cell 1 — load, schema, duplicate và tách X/y.
MLG_BASE_CONFIG = {"n_splits": 5, "shuffle": True, "random_state": 42, "standard_scaler": True, "smote": True, "smote_k_neighbors": 5, "smote_sampling_strategy": 1.0}
mlg_pre_df = pd.read_csv(find_data_file("creditcard.csv"))
mlg_expected_cols = ["Time", *[f"V{i}" for i in range(1, 29)], "Amount", "Class"]
mlg_missing_cols = [col for col in mlg_expected_cols if col not in mlg_pre_df.columns]
mlg_extra_cols = [col for col in mlg_pre_df.columns if col not in mlg_expected_cols]
print("Missing expected columns:", mlg_missing_cols)
print("Extra columns:", mlg_extra_cols)
assert not mlg_missing_cols, f"Thiếu cột bắt buộc: {mlg_missing_cols}"
mlg_dup_mask = mlg_pre_df.duplicated(keep=False)
mlg_dup_count = int(mlg_pre_df.duplicated().sum())
print("Số duplicate hoàn toàn:", mlg_dup_count)
if mlg_dup_count:
    display(mlg_pre_df.loc[mlg_dup_mask, "Class"].value_counts())
mlg_variant_frames = {
    "with_duplicates": mlg_pre_df.reset_index(drop=True).copy(),
    "without_duplicates": mlg_pre_df.drop_duplicates().reset_index(drop=True),
}
mlg_variant_configs = {
    name: {**MLG_BASE_CONFIG, "duplicate_variant": name, "remove_duplicates": name == "without_duplicates"}
    for name in mlg_variant_frames
}
mlg_X_by_variant = {name: frame.drop(columns=["Class"]).copy() for name, frame in mlg_variant_frames.items()}
mlg_y_by_variant = {name: frame["Class"].astype(int).copy() for name, frame in mlg_variant_frames.items()}
mlg_feature_names = mlg_X_by_variant["with_duplicates"].columns.tolist()
for name, frame in mlg_variant_frames.items():
    print(f"{name}: {frame.shape[0]:,} dòng × {frame.shape[1]:,} cột")

In [ ]:
# MLG Cell 2 — Stratified 5-Fold cố định.
mlg_fold_indices_by_variant = {}
mlg_output_dirs_by_variant = {}
for variant_name in ["with_duplicates", "without_duplicates"]:
    config = mlg_variant_configs[variant_name]
    X_variant, y_variant = mlg_X_by_variant[variant_name], mlg_y_by_variant[variant_name]
    skf = StratifiedKFold(n_splits=config["n_splits"], shuffle=config["shuffle"], random_state=config["random_state"])
    fold_indices = list(skf.split(X_variant, y_variant))
    mlg_fold_indices_by_variant[variant_name] = fold_indices
    output_dir = DATASET_OUTPUT_DIRS["MLG_ULB"] / variant_name
    mlg_output_dirs_by_variant[variant_name] = output_dir
    pd.DataFrame(
        [{"parameter": key, "value": value} for key, value in config.items()]
    ).to_csv(output_dir / "config.csv", index=False)
    validation_fold = np.zeros(len(X_variant), dtype=np.int8)
    for fold_number, (_, valid_idx) in enumerate(fold_indices, 1):
        validation_fold[np.asarray(valid_idx)] = fold_number
    pd.DataFrame({
        "source_index": np.arange(len(X_variant)),
        "validation_fold": validation_fold,
    }).to_csv(output_dir / "fold_assignments.csv", index=False)
    print(f"\n{variant_name}")
    for fold_number, (train_idx, valid_idx) in enumerate(fold_indices, 1):
        print(f"Fold {fold_number}: train={len(train_idx):,}, valid={len(valid_idx):,}, train fraud={y_variant.iloc[train_idx].mean():.6%}, valid fraud={y_variant.iloc[valid_idx].mean():.6%}")

In [ ]:
# MLG Cell 3 — scaler và SMOTE chỉ fit trên training fold.
def prepare_mlg_fold(X, y, train_idx, valid_idx, config):
    X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
    y_train, y_valid = y.iloc[train_idx].copy(), y.iloc[valid_idx].copy()
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)
    smote = SMOTE(sampling_strategy=config["smote_sampling_strategy"], k_neighbors=config["smote_k_neighbors"], random_state=config["random_state"])
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
    return {"X_train": X_train_res, "y_train": y_train_res, "X_valid": X_valid_scaled, "y_valid": y_valid.to_numpy(), "scaler": scaler}

In [ ]:
# MLG Cell 4 — xử lý đúng SELECTED_FOLD và chỉ giữ một fold trong output.
MLG_VARIANTS_TO_EXPORT = ["with_duplicates", "without_duplicates"]
MLG_FOLDS_TO_EXPORT = [SELECTED_FOLD]
for variant_name in MLG_VARIANTS_TO_EXPORT:
    if variant_name not in mlg_variant_frames:
        raise ValueError(f"Nhánh duplicate không hợp lệ: {variant_name}")
    X_variant, y_variant = mlg_X_by_variant[variant_name], mlg_y_by_variant[variant_name]
    config = mlg_variant_configs[variant_name]
    fold_indices = mlg_fold_indices_by_variant[variant_name]
    for fold_number in MLG_FOLDS_TO_EXPORT:
        if fold_number not in range(1, 6):
            raise ValueError(f"Fold MLG không hợp lệ: {fold_number}")
        variant_output_dir = DATASET_OUTPUT_DIRS["MLG_ULB"] / variant_name
        variant_output_dir.mkdir(parents=True, exist_ok=True)
        paths = get_fold_csv_paths(variant_output_dir, fold_number)
        if not prepare_single_fold_storage(variant_output_dir, fold_number, paths):
            print(f"Bỏ qua MLG {variant_name} fold {fold_number}: CSV đã hoàn tất.")
            continue
        train_idx, valid_idx = fold_indices[fold_number - 1]
        fold_data = prepare_mlg_fold(X_variant, y_variant, train_idx, valid_idx, config)
        train_df = pd.DataFrame(np.asarray(fold_data["X_train"], dtype=np.float32), columns=mlg_feature_names)
        train_df["Class"] = np.asarray(fold_data["y_train"], dtype=np.int8)
        validation_df = pd.DataFrame(np.asarray(fold_data["X_valid"], dtype=np.float32), columns=mlg_feature_names)
        validation_df["Class"] = np.asarray(fold_data["y_valid"], dtype=np.int8)
        reference_df = pd.DataFrame({"source_index": np.asarray(valid_idx)})
        metadata_df = pd.DataFrame([{
            "dataset": "MLG_ULB", "target": "Class",
            "duplicate_variant": variant_name, "fold": fold_number,
            "train_rows_before_smote": len(train_idx),
            "train_rows_after_smote": len(fold_data["y_train"]),
            "valid_rows": len(valid_idx),
            "train_legitimate_after_smote": int(np.bincount(fold_data["y_train"])[0]),
            "train_fraud_after_smote": int(np.bincount(fold_data["y_train"])[1]),
            "valid_legitimate": int(np.bincount(fold_data["y_valid"])[0]),
            "valid_fraud": int(np.bincount(fold_data["y_valid"])[1]),
        }])
        write_csv_gzip_atomic(train_df, paths["train"])
        write_csv_gzip_atomic(validation_df, paths["validation"])
        reference_df.to_csv(paths["reference"], index=False)
        metadata_df.to_csv(paths["metadata"], index=False)
        pd.DataFrame([{"status": "complete", "fold": fold_number}]).to_csv(paths["success"], index=False)
        print(f"Đã lưu CSV gzip nhánh {variant_name}, fold {fold_number}: {variant_output_dir}")
if MLG_FOLDS_TO_EXPORT:
    for variable_name in ["fold_data", "train_df", "validation_df", "reference_df", "metadata_df"]:
        globals().pop(variable_name, None)
del mlg_pre_df, mlg_variant_frames, mlg_X_by_variant, mlg_y_by_variant
gc.collect()

### Bộ 2: IEEE-FRAUD-DETECTION

In [ ]:
# IEEE Cell 1 — load, LEFT JOIN, target/ID và các cờ reproduction.
IEEE_CONFIG = {"join": "left", "missing_threshold": 0.50, "numeric_fill_value": -999, "categorical_fill_value": "Unknown", "smote_k_neighbors": 5, "smote_sampling_strategy": 1.0, "stratified_kfold": 5, "shuffle": True, "random_state": 42, "treat_coded_numeric_as_categorical": False, "transaction_amt_log": False, "engineer_transaction_dt": False, "use_smotenc": False, "temporal_holdout": False}
ieee_tx = pd.read_csv(find_data_file("train_transaction.csv"))
ieee_identity = pd.read_csv(find_data_file("train_identity.csv"))
ieee_pre_df = ieee_tx.merge(ieee_identity, on="TransactionID", how=IEEE_CONFIG["join"])
assert len(ieee_pre_df) == len(ieee_tx), "LEFT JOIN phải giữ nguyên số transaction."
ieee_y = ieee_pre_df["isFraud"].astype(int).copy()
ieee_transaction_ids = ieee_pre_df["TransactionID"].copy()
ieee_X = ieee_pre_df.drop(columns=["isFraud", "TransactionID"]).copy()
ieee_potential_coded = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
if IEEE_CONFIG["transaction_amt_log"]:
    ieee_X["TransactionAmt_log"] = np.log1p(ieee_X.pop("TransactionAmt"))
if IEEE_CONFIG["engineer_transaction_dt"] and "TransactionDT" in ieee_X.columns:
    sec = ieee_X["TransactionDT"]
    ieee_X["transaction_hour_relative"] = ((sec // 3600) % 24).astype("int16")
    ieee_X["transaction_day_relative"] = (sec // 86400).astype("int32")
    ieee_X["transaction_week_relative"] = (sec // (86400 * 7)).astype("int32")
ieee_categorical_cols = ieee_X.select_dtypes(include=["object", "category"]).columns.tolist()
if IEEE_CONFIG["treat_coded_numeric_as_categorical"]:
    ieee_categorical_cols += [col for col in ieee_potential_coded if col in ieee_X.columns and col not in ieee_categorical_cols]
ieee_numeric_cols = [col for col in ieee_X.columns if col not in ieee_categorical_cols]
print("Transaction:", ieee_tx.shape, "Identity:", ieee_identity.shape, "Merged:", ieee_pre_df.shape)
print("X:", ieee_X.shape, "y:", ieee_y.shape, "Categorical:", len(ieee_categorical_cols), "Numeric:", len(ieee_numeric_cols))

In [ ]:
# IEEE Cell 2 — Stratified 5-Fold trước mọi bước fit preprocessing.
ieee_skf = StratifiedKFold(n_splits=IEEE_CONFIG["stratified_kfold"], shuffle=IEEE_CONFIG["shuffle"], random_state=IEEE_CONFIG["random_state"])
ieee_fold_indices = list(ieee_skf.split(ieee_X, ieee_y))
for fold_number, (train_idx, valid_idx) in enumerate(ieee_fold_indices, 1):
    print(f"Fold {fold_number}: train={len(train_idx):,}, valid={len(valid_idx):,}, train fraud={ieee_y.iloc[train_idx].mean():.6%}, valid fraud={ieee_y.iloc[valid_idx].mean():.6%}")
ieee_output_dir = DATASET_OUTPUT_DIRS["IEEE_CIS"]
save_dataset_definition(ieee_output_dir, IEEE_CONFIG, ieee_fold_indices)

In [ ]:
# IEEE Cell 3 — drop missing theo train, fit preprocessor trên train và SMOTE train.
def get_ieee_drop_cols(X_train, threshold):
    missing_ratio = X_train.isna().mean()
    return missing_ratio[missing_ratio > threshold].index.tolist()

def prepare_ieee_fold(X, y, train_idx, valid_idx, categorical_cols, numeric_cols, config):
    X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
    y_train, y_valid = y.iloc[train_idx].copy(), y.iloc[valid_idx].copy()
    drop_cols = get_ieee_drop_cols(X_train, config["missing_threshold"])
    X_train, X_valid = X_train.drop(columns=drop_cols), X_valid.drop(columns=drop_cols)
    fold_cat = [col for col in categorical_cols if col in X_train.columns]
    fold_num = [col for col in numeric_cols if col in X_train.columns]
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value=config["numeric_fill_value"])), ("scaler", StandardScaler())])
    categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value=config["categorical_fill_value"])), ("onehot", make_dense_one_hot_encoder())])
    preprocessor = ColumnTransformer([("num", numeric_pipe, fold_num), ("cat", categorical_pipe, fold_cat)])
    X_train_proc = preprocessor.fit_transform(X_train)
    X_valid_proc = preprocessor.transform(X_valid)
    smote = SMOTE(sampling_strategy=config["smote_sampling_strategy"], k_neighbors=config["smote_k_neighbors"], random_state=config["random_state"])
    X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)
    return {"X_train": X_train_res, "y_train": y_train_res, "X_valid": X_valid_proc, "y_valid": y_valid.to_numpy(), "drop_cols": drop_cols, "preprocessor": preprocessor}

In [ ]:
# IEEE Cell 4 — xử lý đúng SELECTED_FOLD; kiểm tra RAM trước khi chạy.
IEEE_FOLDS_TO_EXPORT = [SELECTED_FOLD]
IEEE_MAX_ESTIMATED_DENSE_GB = 8.0
for fold_number in IEEE_FOLDS_TO_EXPORT:
    if fold_number not in range(1, 6):
        raise ValueError(f"Fold IEEE không hợp lệ: {fold_number}")
    ieee_output_dir.mkdir(parents=True, exist_ok=True)
    paths = get_fold_csv_paths(ieee_output_dir, fold_number)
    if not prepare_single_fold_storage(ieee_output_dir, fold_number, paths):
        print(f"Bỏ qua IEEE fold {fold_number}: CSV đã hoàn tất.")
        continue
    train_idx, valid_idx = ieee_fold_indices[fold_number - 1]
    preview = ieee_X.iloc[train_idx]
    preview = preview.drop(columns=get_ieee_drop_cols(preview, IEEE_CONFIG["missing_threshold"]))
    preview_cat = [col for col in ieee_categorical_cols if col in preview.columns]
    estimated_features = len([col for col in ieee_numeric_cols if col in preview.columns]) + sum(preview[col].nunique(dropna=False) for col in preview_cat)
    estimated_rows = int(ieee_y.iloc[train_idx].value_counts().max()) * 2
    estimated_gb = estimated_rows * estimated_features * 8 / 1024**3
    print(f"Fold {fold_number} — RAM dense tối thiểu ước lượng: {estimated_gb:.2f} GB")
    if estimated_gb > IEEE_MAX_ESTIMATED_DENSE_GB:
        raise MemoryError(f"Ước lượng {estimated_gb:.2f} GB vượt ngưỡng; không tự đổi One-Hot/SMOTE.")
    fold_data = prepare_ieee_fold(ieee_X, ieee_y, train_idx, valid_idx, ieee_categorical_cols, ieee_numeric_cols, IEEE_CONFIG)
    feature_names = fold_data["preprocessor"].get_feature_names_out()
    train_df = pd.DataFrame(np.asarray(fold_data["X_train"], dtype=np.float32), columns=feature_names)
    train_df["isFraud"] = np.asarray(fold_data["y_train"], dtype=np.int8)
    validation_df = pd.DataFrame(np.asarray(fold_data["X_valid"], dtype=np.float32), columns=feature_names)
    validation_df["isFraud"] = np.asarray(fold_data["y_valid"], dtype=np.int8)
    reference_df = pd.DataFrame({
        "source_index": np.asarray(valid_idx),
        "TransactionID": ieee_transaction_ids.iloc[valid_idx].to_numpy(),
    })
    metadata_df = pd.DataFrame([{
        "dataset": "IEEE_CIS", "target": "isFraud", "fold": fold_number,
        "dropped_columns": "|".join(fold_data["drop_cols"]),
        "estimated_dense_train_gb": estimated_gb,
        "train_rows_after_smote": len(fold_data["y_train"]),
        "valid_rows": len(valid_idx),
    }])
    write_csv_gzip_atomic(train_df, paths["train"])
    write_csv_gzip_atomic(validation_df, paths["validation"])
    reference_df.to_csv(paths["reference"], index=False)
    metadata_df.to_csv(paths["metadata"], index=False)
    pd.DataFrame([{"status": "complete", "fold": fold_number}]).to_csv(paths["success"], index=False)
    print(f"Đã lưu CSV gzip IEEE fold {fold_number}: {ieee_output_dir}")
if IEEE_FOLDS_TO_EXPORT:
    for variable_name in ["fold_data", "train_df", "validation_df", "reference_df", "metadata_df"]:
        globals().pop(variable_name, None)
del ieee_tx, ieee_identity, ieee_pre_df, ieee_X, ieee_y
gc.collect()

### Bộ 3: Sparkov

In [ ]:
# Sparkov Cell 1 — load, loại ID kỹ thuật và áp dụng cờ reproduction.
SPARKOV_CONFIG = {"drop_unnamed_index": True, "drop_trans_num": True, "drop_cc_num": True, "engineer_time_features": True, "engineer_age": False, "drop_high_cardinality_personal": True, "keep_unix_time": True, "engineer_geo_distance": False, "smote_k_neighbors": 5, "smote_sampling_strategy": 1.0, "stratified_kfold": 5, "shuffle": True, "random_state": 42, "use_external_fraud_test": False, "use_smotenc": False}
sparkov_pre_df = pd.read_csv(find_data_file("fraudTrain.csv"))
assert "is_fraud" in sparkov_pre_df.columns
if SPARKOV_CONFIG["drop_unnamed_index"] and "Unnamed: 0" in sparkov_pre_df.columns:
    sparkov_pre_df = sparkov_pre_df.drop(columns=["Unnamed: 0"])
sparkov_transaction_ids = sparkov_pre_df["trans_num"].copy() if "trans_num" in sparkov_pre_df.columns else None
if SPARKOV_CONFIG["drop_trans_num"] and "trans_num" in sparkov_pre_df.columns:
    sparkov_pre_df = sparkov_pre_df.drop(columns=["trans_num"])
if SPARKOV_CONFIG["drop_cc_num"] and "cc_num" in sparkov_pre_df.columns:
    sparkov_pre_df = sparkov_pre_df.drop(columns=["cc_num"])
sparkov_tx_dt = pd.to_datetime(sparkov_pre_df["trans_date_trans_time"], errors="coerce")
if SPARKOV_CONFIG["engineer_age"] and "dob" in sparkov_pre_df.columns:
    sparkov_dob_dt = pd.to_datetime(sparkov_pre_df["dob"], errors="coerce")
    sparkov_pre_df["age"] = (sparkov_tx_dt - sparkov_dob_dt).dt.days / 365.25
    sparkov_pre_df = sparkov_pre_df.drop(columns=["dob"])
if SPARKOV_CONFIG["engineer_time_features"]:
    sparkov_pre_df["transaction_hour"] = sparkov_tx_dt.dt.hour
    sparkov_pre_df["transaction_day"] = sparkov_tx_dt.dt.day
    sparkov_pre_df["transaction_month"] = sparkov_tx_dt.dt.month
    sparkov_pre_df["transaction_weekday"] = sparkov_tx_dt.dt.dayofweek
    sparkov_pre_df["is_weekend"] = (sparkov_tx_dt.dt.dayofweek >= 5).astype("int8")
    sparkov_pre_df = sparkov_pre_df.drop(columns=["trans_date_trans_time"])
if SPARKOV_CONFIG["drop_high_cardinality_personal"]:
    sparkov_pre_df = sparkov_pre_df.drop(columns=[col for col in ["first", "last", "street"] if col in sparkov_pre_df.columns])
if not SPARKOV_CONFIG["keep_unix_time"] and "unix_time" in sparkov_pre_df.columns:
    sparkov_pre_df = sparkov_pre_df.drop(columns=["unix_time"])
if SPARKOV_CONFIG["engineer_geo_distance"]:
    lat1, lon1 = np.radians(sparkov_pre_df["lat"]), np.radians(sparkov_pre_df["long"])
    lat2, lon2 = np.radians(sparkov_pre_df["merch_lat"]), np.radians(sparkov_pre_df["merch_long"])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    sparkov_pre_df["customer_merchant_distance_km"] = 2 * 6371.0 * np.arcsin(np.sqrt(a))
sparkov_y = sparkov_pre_df["is_fraud"].astype(int).copy()
sparkov_X = sparkov_pre_df.drop(columns=["is_fraud"]).copy()
sparkov_categorical_cols = sparkov_X.select_dtypes(include=["object", "category"]).columns.tolist()
sparkov_numeric_cols = [col for col in sparkov_X.columns if col not in sparkov_categorical_cols]
print("X:", sparkov_X.shape, "y:", sparkov_y.shape)
print("Categorical:", sparkov_categorical_cols)
print("Numeric:", sparkov_numeric_cols)

In [ ]:
# Sparkov Cell 2 — Stratified 5-Fold trên fraudTrain.
sparkov_skf = StratifiedKFold(n_splits=SPARKOV_CONFIG["stratified_kfold"], shuffle=SPARKOV_CONFIG["shuffle"], random_state=SPARKOV_CONFIG["random_state"])
sparkov_fold_indices = list(sparkov_skf.split(sparkov_X, sparkov_y))
for fold_number, (train_idx, valid_idx) in enumerate(sparkov_fold_indices, 1):
    print(f"Fold {fold_number}: train={len(train_idx):,}, valid={len(valid_idx):,}, train fraud={sparkov_y.iloc[train_idx].mean():.6%}, valid fraud={sparkov_y.iloc[valid_idx].mean():.6%}")
sparkov_output_dir = DATASET_OUTPUT_DIRS["Sparkov"]
save_dataset_definition(sparkov_output_dir, SPARKOV_CONFIG, sparkov_fold_indices)

In [ ]:
# Sparkov Cell 3 — fit encoder/scaler trên train fold và SMOTE chỉ train.
def prepare_sparkov_fold(X, y, train_idx, valid_idx, numeric_cols, categorical_cols, config):
    X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
    y_train, y_valid = y.iloc[train_idx].copy(), y.iloc[valid_idx].copy()
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value=-999)), ("scaler", StandardScaler())])
    categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")), ("onehot", make_dense_one_hot_encoder())])
    preprocessor = ColumnTransformer([("num", numeric_pipe, numeric_cols), ("cat", categorical_pipe, categorical_cols)])
    X_train_proc = preprocessor.fit_transform(X_train)
    X_valid_proc = preprocessor.transform(X_valid)
    smote = SMOTE(sampling_strategy=config["smote_sampling_strategy"], k_neighbors=config["smote_k_neighbors"], random_state=config["random_state"])
    X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)
    return {"X_train": X_train_res, "y_train": y_train_res, "X_valid": X_valid_proc, "y_valid": y_valid.to_numpy(), "preprocessor": preprocessor}

In [ ]:
# Sparkov Cell 4 — xử lý đúng SELECTED_FOLD; kiểm tra RAM trước khi chạy.
SPARKOV_FOLDS_TO_EXPORT = [SELECTED_FOLD]
SPARKOV_MAX_ESTIMATED_DENSE_GB = 8.0
for fold_number in SPARKOV_FOLDS_TO_EXPORT:
    if fold_number not in range(1, 6):
        raise ValueError(f"Fold Sparkov không hợp lệ: {fold_number}")
    sparkov_output_dir.mkdir(parents=True, exist_ok=True)
    paths = get_fold_csv_paths(sparkov_output_dir, fold_number)
    if not prepare_single_fold_storage(sparkov_output_dir, fold_number, paths):
        print(f"Bỏ qua Sparkov fold {fold_number}: CSV đã hoàn tất.")
        continue
    train_idx, valid_idx = sparkov_fold_indices[fold_number - 1]
    preview = sparkov_X.iloc[train_idx]
    estimated_features = len(sparkov_numeric_cols) + sum(preview[col].nunique(dropna=False) for col in sparkov_categorical_cols)
    estimated_rows = int(sparkov_y.iloc[train_idx].value_counts().max()) * 2
    estimated_gb = estimated_rows * estimated_features * 8 / 1024**3
    print(f"Fold {fold_number} — RAM dense tối thiểu ước lượng: {estimated_gb:.2f} GB")
    if estimated_gb > SPARKOV_MAX_ESTIMATED_DENSE_GB:
        raise MemoryError(f"Ước lượng {estimated_gb:.2f} GB vượt ngưỡng; không tự đổi One-Hot/SMOTE.")
    fold_data = prepare_sparkov_fold(sparkov_X, sparkov_y, train_idx, valid_idx, sparkov_numeric_cols, sparkov_categorical_cols, SPARKOV_CONFIG)
    feature_names = fold_data["preprocessor"].get_feature_names_out()
    train_df = pd.DataFrame(np.asarray(fold_data["X_train"], dtype=np.float32), columns=feature_names)
    train_df["is_fraud"] = np.asarray(fold_data["y_train"], dtype=np.int8)
    validation_df = pd.DataFrame(np.asarray(fold_data["X_valid"], dtype=np.float32), columns=feature_names)
    validation_df["is_fraud"] = np.asarray(fold_data["y_valid"], dtype=np.int8)
    reference_df = pd.DataFrame({"source_index": np.asarray(valid_idx)})
    if sparkov_transaction_ids is not None:
        reference_df["trans_num"] = sparkov_transaction_ids.iloc[valid_idx].to_numpy()
    metadata_df = pd.DataFrame([{
        "dataset": "Sparkov", "target": "is_fraud", "fold": fold_number,
        "estimated_dense_train_gb": estimated_gb,
        "train_rows_after_smote": len(fold_data["y_train"]),
        "valid_rows": len(valid_idx),
    }])
    write_csv_gzip_atomic(train_df, paths["train"])
    write_csv_gzip_atomic(validation_df, paths["validation"])
    reference_df.to_csv(paths["reference"], index=False)
    metadata_df.to_csv(paths["metadata"], index=False)
    pd.DataFrame([{"status": "complete", "fold": fold_number}]).to_csv(paths["success"], index=False)
    print(f"Đã lưu CSV gzip Sparkov fold {fold_number}: {sparkov_output_dir}")
if SPARKOV_FOLDS_TO_EXPORT:
    for variable_name in ["fold_data", "train_df", "validation_df", "reference_df", "metadata_df"]:
        globals().pop(variable_name, None)
del sparkov_pre_df, sparkov_X, sparkov_y
gc.collect()